# CP1 Week 6 -- Loops: Counting Events

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Count threshold crossings in time-series data
2. Detect peaks and valleys
3. Handle edge cases (empty data, single values, all same)
4. Track consecutive streaks above/below thresholds
5. Build event detection into your pipeline

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: What Is an Event?

An **event** is something that happens at a specific point in your data.
Examples:
- Temperature crosses above 60 C (threshold crossing)
- A sensor reading is higher than both its neighbors (peak)
- A value suddenly jumps by more than 20 units (spike)

Detecting events is critical for engineering applications: "How many times
did the motor overheat?" "When did the signal drop?"

### Example 1 -- Threshold crossings

In [ ]:
def count_threshold_crossings(values, threshold):
    """Count how many times values cross above a threshold.

    A crossing happens when we go from below-or-equal to above.
    """
    if len(values) < 2:
        return 0

    crossings = 0
    was_above = values[0] > threshold

    for v in values[1:]:
        is_above = v > threshold
        if is_above and not was_above:
            crossings += 1
        was_above = is_above

    return crossings

# Test
data = [10, 30, 20, 40, 15, 50, 25, 35, 10, 45]
threshold = 25
count = count_threshold_crossings(data, threshold)
print(f"Data: {data}")
print(f"Threshold: {threshold}")
print(f"Crossings above threshold: {count}")

**Expected Output:**
```
Data: [10, 30, 20, 40, 15, 50, 25, 35, 10, 45]
Threshold: 25
Crossings above threshold: 4
```

### Example 2 -- Finding peaks

In [ ]:
def find_peaks(values):
    """Find indices where value is higher than both neighbors."""
    if len(values) < 3:
        return []
    peaks = []
    for i in range(1, len(values) - 1):
        if values[i] > values[i-1] and values[i] > values[i+1]:
            peaks.append(i)
    return peaks

data = [10, 30, 20, 40, 15, 50, 25, 35, 10, 45]
peaks = find_peaks(data)
print(f"Data:  {data}")
print(f"Peaks at indices: {peaks}")
peak_values = [data[i] for i in peaks]
print(f"Peak values: {peak_values}")

**Expected Output:**
```
Data:  [10, 30, 20, 40, 15, 50, 25, 35, 10, 45]
Peaks at indices: [1, 3, 5, 7]
Peak values: [30, 40, 50, 35]
```

### Example 3 -- Edge cases

In [ ]:
def safe_count_events(values, threshold):
    """Count threshold crossings with edge case handling."""
    if not values:
        print("  Warning: empty data")
        return 0
    if len(values) < 2:
        print("  Warning: need at least 2 values")
        return 0
    if all(v == values[0] for v in values):
        print("  Warning: all values identical")
        return 0
    return count_threshold_crossings(values, threshold)

# Test edge cases
print("Edge case tests:")
print("  Empty:", safe_count_events([], 10))
print("  Single:", safe_count_events([5], 10))
print("  Same:", safe_count_events([5, 5, 5], 10))
print("  Normal:", safe_count_events([5, 15, 5, 15], 10))

**Expected Output:**
```
Edge case tests:
  Warning: empty data
  Empty: 0
  Warning: need at least 2 values
  Single: 0
  Warning: all values identical
  Same: 0
  Normal: 2
```

### Try It Yourself #1

In [ ]:
# TODO: Write find_valleys(values) -- the opposite of find_peaks.
# A valley is lower than BOTH neighbors.

def find_valleys(values):
    pass  # your code here

data = [30, 10, 25, 5, 35, 15, 40, 20, 50]
valleys = find_valleys(data)
print(f"Data: {data}")
print(f"Valleys at: {valleys}")

### Debugging Tip

When debugging event detection, **print the state at each step**:
```python
for i, v in enumerate(values):
    was = "above" if was_above else "below"
    now = "above" if v > threshold else "below"
    crossed = "CROSSED!" if now != was else ""
    print(f"  [{i}] v={v}, was={was}, now={now} {crossed}")
```
This makes it easy to see exactly where crossings happen.

### Example 4 -- Longest streak above threshold

In [ ]:
def longest_streak(values, threshold):
    """Find the longest consecutive run of values above threshold."""
    max_streak = 0
    current_streak = 0

    for v in values:
        if v > threshold:
            current_streak += 1
            if current_streak > max_streak:
                max_streak = current_streak
        else:
            current_streak = 0

    return max_streak

data = [10, 30, 35, 40, 20, 50, 55, 60, 65, 10, 30]
threshold = 25
streak = longest_streak(data, threshold)
print(f"Longest streak above {threshold}: {streak} consecutive values")

**Expected Output:**
```
Longest streak above 25: 4 consecutive values
```

### Example 5 -- Spike detection

In [ ]:
def find_spikes(values, jump_threshold):
    """Find sudden jumps between consecutive readings."""
    spikes = []
    for i in range(1, len(values)):
        jump = abs(values[i] - values[i-1])
        if jump > jump_threshold:
            spikes.append({
                "index": i,
                "from": values[i-1],
                "to": values[i],
                "jump": round(jump, 2),
            })
    return spikes

data = [20, 22, 21, 80, 23, 22, 90, 25, 24, 23]
spikes = find_spikes(data, 20)
print(f"Data: {data}")
print(f"Spikes (jump > 20):")
for s in spikes:
    direction = "UP" if s["to"] > s["from"] else "DOWN"
    print(f"  Index {s['index']}: {s['from']} -> {s['to']} ({direction}, jump={s['jump']})")

**Expected Output:**
```
Data: [20, 22, 21, 80, 23, 22, 90, 25, 24, 23]
Spikes (jump > 20):
  Index 3: 21 -> 80 (UP, jump=59)
  Index 4: 80 -> 23 (DOWN, jump=57)
  Index 6: 22 -> 90 (UP, jump=68)
  Index 7: 90 -> 25 (DOWN, jump=65)
```

### Example 6 -- Complete event summary

In [ ]:
def event_summary(values, threshold):
    """Generate a complete event summary for a dataset."""
    if len(values) < 2:
        return "Insufficient data"

    crossings = count_threshold_crossings(values, threshold)
    peaks = find_peaks(values)
    streak = longest_streak(values, threshold)
    spikes = find_spikes(values, 15)

    above_count = sum(1 for v in values if v > threshold)
    below_count = len(values) - above_count

    print(f"=== Event Summary (threshold={threshold}) ===")
    print(f"  Total readings:     {len(values)}")
    print(f"  Above threshold:    {above_count} ({above_count/len(values)*100:.1f}%)")
    print(f"  Below threshold:    {below_count} ({below_count/len(values)*100:.1f}%)")
    print(f"  Threshold crossings: {crossings}")
    print(f"  Peaks detected:     {len(peaks)}")
    print(f"  Longest streak:     {streak} consecutive values")
    print(f"  Spikes (>15):       {len(spikes)}")

    return {
        "crossings": crossings,
        "peaks": len(peaks),
        "streak": streak,
        "spikes": len(spikes),
    }

data = [10, 30, 20, 40, 15, 50, 25, 35, 10, 45, 55, 60, 40, 20]
result = event_summary(data, 25)

### Why This Matters for Your Pipeline

Event detection answers the questions engineers actually care about:
- "How many times did the motor overheat today?" (threshold crossings)
- "When were the peak temperatures?" (peak detection)
- "How long did the longest overheating episode last?" (streak length)
- "Were there any sudden failures?" (spike detection)

These metrics go into your `analyze()` function's output and your report.

---
## Key Takeaways -- Week 6

1. **Threshold crossings** track state transitions (below->above)
2. **Peaks** are local maxima (higher than both neighbors)
3. **Always handle edge cases**: empty, single value, all identical
4. **Streaks** count consecutive values meeting a condition
5. **Spikes** detect sudden jumps between consecutive readings
6. **Event summaries** combine multiple detection methods into one report

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What is a threshold crossing?
# R2: What is a peak in time-series data?
# R3: Name 3 edge cases you should always handle.
# R4: How do you track a "streak" using a loop?

### Practice (P1-P5)

In [ ]:
# P1: Count how many times temperature goes ABOVE and BELOW threshold.
temps = [20, 35, 28, 40, 22, 38, 15, 42, 30]
threshold = 30


In [ ]:
# P2: Find all valleys (lower than both neighbors).
data = [30, 10, 25, 5, 35, 15, 40]


In [ ]:
# P3: Find the longest streak of values BELOW a threshold.
data = [50, 20, 15, 10, 30, 5, 8, 12, 40, 3]
threshold = 25


In [ ]:
# P4: Write a function that finds ALL streaks above threshold.
# Return a list of (start_index, length) tuples.


In [ ]:
# P5: Detect "spikes" -- sudden jumps of more than X between consecutive readings.
data = [20, 22, 21, 80, 23, 22, 90, 25]


### Challenge (C1-C3)

In [ ]:
# C1: Write a "state machine" that tracks NORMAL/WARNING/CRITICAL states.
# Count how many transitions occur between states.


In [ ]:
# C2: Detect "oscillation" -- when values rapidly alternate above/below threshold.


In [ ]:
# C3: Write a complete event_summary() function that returns:
# crossings, peaks, valleys, longest_streak, spike_count


### Mini-Project

In [ ]:
# M1: Sensor Event Monitor
# Given 50+ readings, produce a complete event report:
# - Threshold crossings (count, timestamps)
# - Peaks and valleys (indices, values)
# - Streaks above threshold (start, length)
# - Spikes (location, magnitude)
# Format as a readable text report.

import random
random.seed(42)
data = [20 + random.gauss(0, 10) for _ in range(50)]
# Add some events
data[15] = 60   # spike
data[16] = 55
data[17] = 50
data[30] = -5   # anomaly


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)